In [15]:
vocab_size = 50257 # Vocabulary size
context_length = 1024 # Context length
emb_dim = 768 # Embedding dimension
n_heads = 12 # Number of attention heads
n_layers = 12 # Number of layers
drop_rate = 0.1 # Dropout rate

In [17]:
class DummyTransformerBlock(nn.Module):
    def __init__(self, vocab_size, context_length, emb_dim, n_heads, n_layers, drop_rate):
        super().__init__()
    def forward(self, x):
        return x
class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
    def forward(self, x):
        return x

In [22]:
import torch
import torch.nn as nn

class DummyGPTModel(nn.Module):
    def __init__(self, vocab_size, context_length, emb_dim, n_heads, n_layers, drop_rate):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.pos_embedding = nn.Embedding(vocab_size, emb_dim)
        self.transformer = DummyTransformerBlock(vocab_size, context_length, emb_dim, n_heads, n_layers, drop_rate)
        self.dropout = nn.Dropout(drop_rate)
        self.ff_layer = nn.Linear(emb_dim, vocab_size)
        self.layer_normalization = DummyLayerNorm(emb_dim)

        
    def forward(self, inputs):
        emb = self.embedding(inputs)
        pos_embedding = self.pos_embedding(torch.arange(inputs.shape[1]))
        x = emb + pos_embedding

        self.dropout(x)

        x = self.transformer(emb)
        x = self.layer_normalization(x)

        return self.ff_layer(x)


In [10]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [23]:
torch.manual_seed(123)
model = DummyGPTModel(vocab_size, context_length, emb_dim, n_heads, n_layers, drop_rate)
logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

Output shape: torch.Size([2, 4, 50257])
tensor([[[-0.8542, -0.8767,  1.1769,  ..., -0.2195,  0.1865,  0.3157],
         [-0.3141, -1.1789,  0.0725,  ..., -0.1020, -0.4723,  0.6295],
         [ 0.4560, -0.4309, -0.5191,  ..., -0.3051,  0.7951, -0.2243],
         [ 0.1672,  0.8893,  0.1453,  ...,  0.2198, -0.5952, -0.0034]],

        [[-0.8542, -0.8767,  1.1769,  ..., -0.2195,  0.1865,  0.3157],
         [-0.2067,  0.0408, -0.0612,  ..., -0.1211, -0.4266,  0.7470],
         [ 0.2454,  0.4970, -0.1554,  ..., -0.5173, -1.0216, -0.3721],
         [ 0.5331,  0.3477,  0.5517,  ...,  0.6545, -0.3696,  0.0160]]],
       grad_fn=<ViewBackward0>)
